# Vehicle Specification Extraction Pipeline (Hugging Face Mistral AI)

This notebook demonstrates the end-to-end pipeline for extracting structured vehicle specifications (torque values, fluid capacities, dimensions, part numbers) from an automotive service manual PDF (`sample-service-manual 1.pdf`, Ford F-150 Workshop Manual) using **Hugging Face Mistral AI** (`mistralai/Mistral-7B-Instruct-v0.3`).

### Pipeline Overview:
1. **PDF Text Extraction:** Table-aware parsing using PyMuPDF, stripping repetitive header/footer noise.
2. **Logical Chunking:** Section- and page-aware chunking preserving specification table rows.
3. **Dense Embeddings & FAISS Index:** Local L2-normalized vectors via `all-MiniLM-L6-v2` and FAISS `IndexFlatIP`.
4. **Semantic Retrieval:** Top-$k$ similarity search retrieving grounded source context.
5. **Mistral AI Extraction & Validation:** Hugging Face serverless `InferenceClient` with strict Pydantic validation outputting JSON or CSV.

In [ ]:
import sys
from pathlib import Path

# Add workspace root to Python path
workspace_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from src.pdf_extractor import PDFExtractor
from src.chunker import TextChunker
from src.embeddings import get_embedder
from src.retriever import FAISSRetriever
from src.specification_extractor import SpecificationExtractor, VehicleSpecification
from src.pipeline import SpecificationPipeline

## 1. Inspect Vector Index Statistics
The 852-page Ford F-150 service manual has already been extracted, chunked, and indexed into `data/faiss_index.bin`.

In [ ]:
pipeline = SpecificationPipeline(index_dir=workspace_root / 'data')
pipeline.show_info()

## 2. PDF Extraction & Table Preservation Demonstration
Let's inspect how Page 636 (Front Disc Brake Specifications) is extracted with table structure intact.

In [ ]:
extractor = PDFExtractor()
pages, summary = extractor.extract_document(workspace_root / 'sample-service-manual 1.pdf', max_pages=636)
page_636 = next(p for p in pages if p.page_number == 636)

print(f"Page {page_636.page_number} | Section: {page_636.section}")
print("=" * 60)
print(page_636.text[:800])

## 3. Semantic Retrieval for Automotive Specification Queries
Retrieving the top relevant chunks for vehicle specification queries.

In [ ]:
query = "What is the torque specification for the brake caliper bolts?"
results = pipeline.retriever.retrieve(query, top_k=3)

print(f"Query: {query}\n")
for r in results:
    print(f"[Rank {r.rank}] Score: {r.score:.4f} | Page {r.page_number} | Section: {r.section}")
    print(f"{r.text[:300]}...\n")

In [ ]:
fluid_query = "What is the fill capacity for High Performance DOT 3 Brake Fluid?"
fluid_results = pipeline.retriever.retrieve(fluid_query, top_k=2)

print(f"Query: {fluid_query}\n")
for r in fluid_results:
    print(f"[Rank {r.rank}] Score: {r.score:.4f} | Page {r.page_number} | Section: {r.section}")
    print(f"{r.text[:300]}...\n")

## 4. Structured Specification Validation & Export
Using Pydantic validation to enforce the required assignment schema and export to JSON or CSV.

In [ ]:
specs = [
    VehicleSpecification(
        component="Brake caliper flow bolt",
        spec_type="Torque",
        value="35",
        unit="Nm",
        page_number=636,
        source_excerpt="Brake caliper flow bolt | 35 | 26 | — |"
    ),
    VehicleSpecification(
        component="Brake caliper anchor plate bolts",
        spec_type="Torque",
        value="250",
        unit="Nm",
        page_number=636,
        source_excerpt="Brake caliper anchor plate bolts | 250 | 184 | — |"
    ),
    VehicleSpecification(
        component="High Performance DOT 3 Brake Fluid",
        spec_type="Fluid Capacity",
        value="1,420",
        unit="ml",
        page_number=636,
        source_excerpt="1,420 ml (3 pt)"
    )
]

print("=== REQUIRED CORE JSON FORMAT ===")
print(SpecificationExtractor.to_json(specs, include_metadata=False))

print("\n=== FULL JSON WITH TRACEABILITY METADATA ===")
print(SpecificationExtractor.to_json(specs, include_metadata=True))

print("\n=== CSV FORMAT ===")
print(SpecificationExtractor.to_csv(specs, include_metadata=True))

## 5. Live Hugging Face Mistral AI Extraction
When `HF_TOKEN` is configured in `.env`, calling `pipeline.run_query(...)` connects directly to `mistralai/Mistral-7B-Instruct-v0.3` via Hugging Face's serverless Inference API.

In [ ]:
import os
if os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    pipeline.run_query("What is the torque specification for the brake caliper bolts?", top_k=3)
else:
    print("HF_TOKEN is not configured in this environment.")
    print("To run live LLM extraction with Mistral AI, set HF_TOKEN in your .env file.")